# Feature Engineering & Data Integration

This notebook merges and prepares user, transaction, and product data for analysis.

## 1. Setup & Initialization

In [30]:
import pandas as pd
from pathlib import Path
import os

In [31]:
BASE_DIR = Path.cwd().parent

SRC_PATH   = BASE_DIR / "Data" / "processed"

In [32]:
df_trans = pd.read_csv(SRC_PATH / "df_transactions.csv", encoding="utf-8-sig")
df_users = pd.read_csv(SRC_PATH / "df_users.csv", encoding="utf-8-sig")
df_commission = pd.read_csv(SRC_PATH / "df_commission.csv", encoding="utf-8-sig")

In [33]:
df_trans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13494 entries, 0 to 13493
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   user_id          13494 non-null  int64 
 1   order_id         13494 non-null  int64 
 2   Date             13494 non-null  object
 3   Amount           13494 non-null  int64 
 4   Merchant_id      13494 non-null  int64 
 5   Purchase_status  13494 non-null  int64 
dtypes: int64(5), object(1)
memory usage: 632.7+ KB


## 2. Integrating Products & Revenue Calculation

Merging the transactions array with product lookup data to calculate the actual revenue for each transaction, based on the percentage rate.

In [34]:
df_trans = df_trans.merge(
    df_commission, on="Merchant_id", how="left"
)

df_trans["Revenue"] = df_trans["Amount"] * (df_trans["Rate_pct"] / 100 )

## 3. Date Transformations & Temporal Features

Converting `Date` and `First_tran_date` columns to standard datetime formats and calculating aggregates like the most profitable month and day.

In [35]:
df_trans['Date'] = pd.to_datetime(df_trans['Date'])


In [36]:
df_users['First_tran_date'] = pd.to_datetime(df_users['First_tran_date'])


### Calculating most profitable month, total for month

In [37]:
monthly_rev = df_trans.groupby(df_trans["Date"].dt.to_period("M"))["Revenue"].sum()

most_profitable_month = monthly_rev.idxmax()
total_for_month = monthly_rev.max()

print(f"Best month: {most_profitable_month}")
print(f"Revenue: {total_for_month:,.0f}")

Best month: 2020-09
Revenue: 1,702,200


In [38]:
weekday_avg = (
    df_trans
    .groupby(df_trans["Date"].dt.day_name())["Revenue"]
    .mean()
)
most_profitable_day = weekday_avg.idxmax()
least_profitable_day = weekday_avg.idxmin()

print(f"Most profitable day:{most_profitable_day}")
print(f"Least profitable day:{least_profitable_day}")

Most profitable day:Wednesday
Least profitable day:Sunday


### Check info

In [39]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13390 entries, 0 to 13389
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   user_id          13390 non-null  int64         
 1   First_tran_date  13390 non-null  datetime64[ns]
 2   Location         13390 non-null  object        
 3   Age              13390 non-null  object        
 4   Gender           13390 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 523.2+ KB


In [40]:
df_trans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13494 entries, 0 to 13493
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   user_id          13494 non-null  int64         
 1   order_id         13494 non-null  int64         
 2   Date             13494 non-null  datetime64[ns]
 3   Amount           13494 non-null  int64         
 4   Merchant_id      13494 non-null  int64         
 5   Purchase_status  13494 non-null  int64         
 6   Merchant_name    13494 non-null  object        
 7   Rate_pct         13494 non-null  int64         
 8   Revenue          13494 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(6), object(1)
memory usage: 948.9+ KB


### Merge dataframe

In [41]:
df_trans_tx = df_trans.merge(
    df_users,
    on="user_id",
    how="left"
)

In [42]:
df_trans_tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13494 entries, 0 to 13493
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   user_id          13494 non-null  int64         
 1   order_id         13494 non-null  int64         
 2   Date             13494 non-null  datetime64[ns]
 3   Amount           13494 non-null  int64         
 4   Merchant_id      13494 non-null  int64         
 5   Purchase_status  13494 non-null  int64         
 6   Merchant_name    13494 non-null  object        
 7   Rate_pct         13494 non-null  int64         
 8   Revenue          13494 non-null  float64       
 9   First_tran_date  13494 non-null  datetime64[ns]
 10  Location         13494 non-null  object        
 11  Age              13494 non-null  object        
 12  Gender           13494 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(6), object(4)
memory usage: 1.3+ MB


## 4. Feature Engineering: Categorization & User Type

Optimizing data types by converting age ranges to categories. We also engineer a new feature, `Type_user`, which flags whether the transaction occurred in the same month as the user's first transaction (New vs Current user).

In [43]:
df_trans_tx["Age"] = df_trans_tx["Age"].astype("category")
df_trans_tx["Age"].value_counts()

Age
23_to_27    3430
28_to_32    2816
unknown     2529
33_to_37    1753
18_to_22    1704
>37         1262
Name: count, dtype: int64

In [44]:
df_trans_tx["Type_user"] = (
    df_trans_tx["Date"].dt.to_period("M") ==
    df_trans_tx["First_tran_date"].dt.to_period("M")
)

df_trans_tx["Type_user"] = df_trans_tx["Type_user"].map({
    True: "New",
    False: "Current"
})

In [45]:
df_trans_tx[df_trans_tx["Type_user"] == "New"] \
    .sort_values(["user_id","Date"]) \
    .head(20)

,user_id,order_id,Date,Amount,Merchant_id,Purchase_status,Merchant_name,Rate_pct,Revenue,First_tran_date,Location,Age,Gender,Type_user
9747,644824,7362695016,2020-09-29,20000,12,0,Viettel,2,400.0,2020-09-29,Other Cities,unknown,MALE,New
13285,967948,8672800182,2020-12-28,200000,12,1,Viettel,2,4000.0,2020-12-10,HCMC,33_to_37,FEMALE,New
1693,2174908,4856496186,2020-02-22,100000,12,0,Viettel,2,2000.0,2020-02-22,Other Cities,unknown,FEMALE,New
3639,2338603,5378659071,2020-04-18,100000,14,0,Vinaphone,4,4000.0,2020-04-06,Other Cities,28_to_32,FEMALE,New
11084,2606556,7873025220,2020-11-02,10000,12,0,Viettel,2,200.0,2020-11-02,Other Cities,18_to_22,FEMALE,New
3890,2930619,5448895812,2020-04-26,20000,14,0,Vinaphone,4,800.0,2020-04-26,Other Cities,unknown,FEMALE,New
565,3002366,4345423020,2020-01-19,20000,13,0,Mobifone,3,600.0,2020-01-16,Other Cities,unknown,MALE,New
11904,3139148,8178022062,2020-11-24,50000,13,0,Mobifone,3,1500.0,2020-11-02,HCMC,33_to_37,MALE,New
8150,3359053,6761897644,2020-08-20,100000,12,0,Viettel,2,2000.0,2020-08-10,Other Cities,unknown,FEMALE,New
8328,3373534,6821951120,2020-08-25,100000,12,0,Viettel,2,2000.0,2020-08-17,Other Cities,unknown,MALE,New


In [46]:
df_trans_tx.dtypes

user_id                     int64
order_id                    int64
Date               datetime64[ns]
Amount                      int64
Merchant_id                 int64
Purchase_status             int64
Merchant_name              object
Rate_pct                    int64
Revenue                   float64
First_tran_date    datetime64[ns]
Location                   object
Age                      category
Gender                     object
Type_user                  object
dtype: object

## 5. Data Export

Saving the integrated and transformed dataset into the processed data folder for further analysis or modeling.

In [48]:
df_trans_tx.to_csv(f"{SRC_PATH}/final.csv",index=False,encoding= 'utf-8-sig')